In [1]:
import os
os.environ["PYSPARK_PYTHON"] = r"C:\Users\Ben\AppData\Local\Programs\Python\Python310\python.exe"
os.environ["PYSPARK_DRIVER_PYTHON"] = r"C:\Users\Ben\AppData\Local\Programs\Python\Python310\python.exe"
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["hadoop.home.dir"] = r"C:\hadoop"

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    broadcast, sum, avg, count, col, round, rand, 
    lit, concat, when, month, year
)
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("Week2_Day2_BroadcastJoins") \
    .master("local[*]") \
    .getOrCreate()

In [3]:
import random

# ── Small lookup table: 50 regions ──
region_data = [(i, f"Region_{i}", random.choice(["North", "South", "East", "West"])) 
               for i in range(1, 51)]

region_schema = StructType([
    StructField("region_id", IntegerType(), False),
    StructField("region_name", StringType(), False),
    StructField("zone", StringType(), False)
])

regions_df = spark.createDataFrame(region_data, schema=region_schema)
regions_df.show(10)

+---------+-----------+-----+
|region_id|region_name| zone|
+---------+-----------+-----+
|        1|   Region_1| East|
|        2|   Region_2| West|
|        3|   Region_3| East|
|        4|   Region_4|South|
|        5|   Region_5|South|
|        6|   Region_6|North|
|        7|   Region_7| East|
|        8|   Region_8| East|
|        9|   Region_9| East|
|       10|  Region_10|South|
+---------+-----------+-----+
only showing top 10 rows



In [4]:
# ── Large fact table: 1M orders ──
orders_df = spark.range(0, 1_000_000) \
    .withColumn("order_id", col("id")) \
    .withColumn("region_id", (rand() * 50 + 1).cast(IntegerType())) \
    .withColumn("order_value", round(rand() * 500 + 10, 2)) \
    .withColumn("revenue", round(col("order_value") * (rand() * 0.4 + 0.6), 2)) \
    .withColumn("order_month", (rand() * 12 + 1).cast(IntegerType())) \
    .drop("id")

orders_df.show(10)
print(f"Orders count: {orders_df.count()}")

+--------+---------+-----------+-------+-----------+
|order_id|region_id|order_value|revenue|order_month|
+--------+---------+-----------+-------+-----------+
|       0|        1|     470.98| 453.89|          6|
|       1|       26|     241.17| 154.43|          4|
|       2|        7|      22.46|  18.44|          5|
|       3|       41|     401.43| 333.37|          7|
|       4|       32|     181.56| 136.89|          5|
|       5|       46|     500.71| 486.91|         11|
|       6|        5|     213.94|  134.0|          7|
|       7|        3|     174.26| 164.75|          8|
|       8|       43|      49.85|  32.37|          8|
|       9|       25|      289.4| 244.48|          9|
+--------+---------+-----------+-------+-----------+
only showing top 10 rows

Orders count: 1000000


In [5]:
# Quick sanity checks
print(f"Orders partitions: {orders_df.rdd.getNumPartitions()}")
print(f"Regions partitions: {regions_df.rdd.getNumPartitions()}")
print(f"Regions count: {regions_df.count()}")

Orders partitions: 4
Regions partitions: 4
Regions count: 50


In [10]:
import time

# Disable auto-broadcast for fair comparison
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

# ── Shuffle Join Timing ──
start = time.time()
result1 = orders_df.join(regions_df, "region_id")
result1.count()  # Force execution (action)
shuffle_time = time.time() - start
print(f"Shuffle Join: {shuffle_time:.2f} seconds")

# ── Broadcast Join Timing ──
start = time.time()
result2 = orders_df.join(broadcast(regions_df), "region_id")
result2.count()  # Force execution
broadcast_time = time.time() - start
print(f"Broadcast Join: {broadcast_time:.2f} seconds")

print(f"Speedup: {shuffle_time / broadcast_time:.1f}x faster")

# Reset threshold
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10485760)

Shuffle Join: 6.47 seconds
Broadcast Join: 4.43 seconds
Speedup: 1.46x faster


In [11]:
# First, join the tables (with broadcast since regions is small)
enriched_orders = orders_df.join(broadcast(regions_df), "region_id")

enriched_orders.show(5)

+---------+--------+-----------+-------+-----------+-----------+----+
|region_id|order_id|order_value|revenue|order_month|region_name|zone|
+---------+--------+-----------+-------+-----------+-----------+----+
|        1|       0|     470.98| 453.89|          6|   Region_1|East|
|       26|       1|     241.17| 154.43|          4|  Region_26|East|
|        7|       2|      22.46|  18.44|          5|   Region_7|East|
|       41|       3|     401.43| 333.37|          7|  Region_41|West|
|       32|       4|     181.56| 136.89|          5|  Region_32|East|
+---------+--------+-----------+-------+-----------+-----------+----+
only showing top 5 rows

